In [17]:
import pandas
import plotly.express as px
import plotly.graph_objects as go

# Load data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Only keep rows for desired kmer
kmer = 55
df = df[df["kmer_size"] == kmer].copy()

# Ensure we don't drop dict-typed columns (like madb_time) prematurely
# Only drop rows where cogent3_time or score are missing
df = df[
    df["cogent3_time"].notnull() &
    df["cogent3_score"].notnull() &
    df["madb_time"].notnull() &  # keep the nested dict
    df["madb_score"].notnull()
].copy()

# Add cycle status for plotting
df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})

# Plot config
species_colors = {
    "Chimpanzee": "blue",
    "Gorilla": "green",
    "Macaque": "red"
}
symbol_sequence = {
    "has_cycles": "x",
    "no_cycles": "circle"
}

In [18]:
# Further filter to rows where MADB time and Cogent3 pd are available
df_filtered = df[
    df["madb_time"].notnull() &
    df["cogent3_pd"].notnull()
].copy()

# Plot
fig = px.scatter(
    df_filtered,
    x="cogent3_pd",
    y="madb_time",
    color="species",
    symbol="cycle_status",
    color_discrete_map=species_colors,
    symbol_map=symbol_sequence,
    hover_data=["unique_id"],
)

fig.update_traces(marker=dict(size=10), showlegend=False)

fig.update_layout(
    width=600,
    height=600,
    margin=dict(l=40, r=20, t=20, b=40),
    xaxis_title="Pairwise divergence (Cogent3)",
    yaxis_title="MADB execution time (s)",
    xaxis=dict(range=[0, df_filtered["cogent3_pd"].max() + 0.01]),
    yaxis=dict(range=[0, df_filtered["madb_time"].max() * 1.1]),
    title=f"MADB Execution Time vs Sequence Divergence (k={kmer})"
)

fig.show()
fig.write_image(f"../figures/madb_time_vs_cogent3_pd_k{kmer}.pdf")


In [19]:
# Filter to rows with valid MADB time
df_violin = df[df["madb_time"].notnull()].copy()

# Create violin plot
fig = px.violin(
    df_violin,
    y="species",
    x="madb_time",
    color="species",
    orientation="h",
    color_discrete_map=species_colors,
    box=True,  # include mini-boxplot inside
    points="all",  # show individual points
)

# Customize layout
fig.update_traces(showlegend=False)

fig.update_layout(
    width=600,
    height=400,
    margin=dict(l=40, r=20, t=20, b=40),
    xaxis_title="MADB execution time (s)",
    yaxis_title="",
    title=f"Execution Time Distribution by Species (k={kmer})"
)

fig.show()


In [20]:
import pandas
import plotly.express as px

# Load data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Only keep rows for desired kmer
kmer = 55
df = df[df["kmer_size"] == kmer].copy()

# Drop rows with missing timing data
df = df[
    df["cogent3_time"].notnull() &
    df["madb_time"].notnull()
].copy()

# Compute the ratio
df["madb_vs_cogent3_ratio"] = df["madb_time"] / df["cogent3_time"]

# Filter out infinite or unreasonable ratios
df = df[df["madb_vs_cogent3_ratio"].between(0, 10)]

# Plot violin of time ratio by species
fig = px.violin(
    df,
    y="species",
    x="madb_vs_cogent3_ratio",
    color="species",
    color_discrete_map={
        "Chimpanzee": "blue",
        "Gorilla": "green",
        "Macaque": "red"
    },
    points="all",  # show data points
    box=False
)

fig.update_layout(
    title="Relative Performance: MADB vs Cogent3 (Execution Time Ratio)",
    xaxis_title="MADB_time / Cogent3_time",
    yaxis_title="Species",
    width=600,
    height=400
)

fig.show()


In [21]:
import pandas
import plotly.express as px

# Load data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Choose k-mer and filter
kmer = 55
df = df[df["kmer_size"] == kmer].copy()

# Ensure necessary data is present
df = df[
    df["cogent3_pd"].notnull() &
    df["madb_bubbles"].notnull()
].copy()

df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})

# Plot: Bubble count vs sequence divergence
fig = px.scatter(
    df,
    x="cogent3_pd",
    y="madb_bubbles",
    color="species",
    color_discrete_map={
        "Chimpanzee": "blue",
        "Gorilla": "green",
        "Macaque": "red"
    },
    symbol="cycle_status",
    symbol_map={"has_cycles": "x", "no_cycles": "circle"},
    hover_data=["unique_id"]
)

fig.update_layout(
    title=f"Bubble Count vs Sequence Divergence (k={kmer})",
    xaxis_title="Pairwise Distance (Cogent3)",
    yaxis_title="Number of Bubbles",
    width=600,
    height=500
)

fig.show()


In [35]:
import pandas
import plotly.express as px

# Load and filter
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
kmer = 55
df = df[df["kmer_size"] == kmer].copy()
df = df[df["cogent3_pd"].notnull() & df["madb_cycles"].notnull()].copy()

# Bin cogent3_pd into string-labeled bins for plotting
df["pd_bin"] = pandas.cut(df["cogent3_pd"], bins=10).astype(str)

# Group and compute proportions
grouped = df.groupby("pd_bin")["madb_cycles"].agg(
    total="count",
    with_cycles="sum"
).reset_index()
grouped["proportion_with_cycles"] = grouped["with_cycles"] / grouped["total"]

# Plot
fig = px.bar(
    grouped,
    x="pd_bin",
    y="proportion_with_cycles",
    labels={
        "pd_bin": "Pairwise Distance Bin",
        "proportion_with_cycles": "Proportion with Cycles"
    },
    title=f"Cycle Frequency vs Sequence Divergence (k={kmer})"
)

fig.update_layout(
    yaxis=dict(range=[0, 1]),
    width=700,
    height=400,
    margin=dict(t=40, b=40, l=60, r=20)
)

fig.show()


In [12]:
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Load and filter
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
kmer = 15
df = df[df["kmer_size"] == kmer].copy()
df = df[df["cogent3_pd"].notnull() & df["madb_cycles"].notnull()].copy()

# Plot config
species_colors = {
    "Chimpanzee": "blue",
    "Gorilla": "green",
    "Macaque": "red"
}
symbol_sequence = {
    "has_cycles": "x",
    "no_cycles": "circle"
}

# Calculate seconds per kbp
df["madb_sec_per_kbp"] = df["madb_time"] / (df["seq_length"] / 1000)

# Filter to non-null values
filtered_df = df[
    df["cogent3_pd"].notnull() &
    df["madb_sec_per_kbp"].notnull()
].copy()

fig = go.Figure()

for species in ["Chimpanzee", "Macaque"]:
    species_df = filtered_df[filtered_df["species"] == species]
    x = species_df["cogent3_pd"].values.reshape(-1, 1)
    y = species_df["madb_sec_per_kbp"].values

    # Regression
    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    slope = model.coef_[0]
    r2 = r2_score(y, y_pred)

    # Scatter points
    fig.add_trace(go.Scatter(
        x=species_df["cogent3_pd"],
        y=species_df["madb_sec_per_kbp"],
        mode="markers",
        marker=dict(color=species_colors[species]),
        name=f"{species} (slope={slope:.3f}, R²={r2:.3f})"
    ))

    # Regression line
    x_range = pandas.Series(sorted(species_df["cogent3_pd"].unique()))
    fig.add_trace(go.Scatter(
        x=x_range,
        y=model.predict(x_range.values.reshape(-1, 1)),
        mode="lines",
        line=dict(dash="dot", color=species_colors[species]),
        showlegend=False
    ))

fig.update_layout(
    title=f"MADB Seconds per kbp vs Divergence (k={kmer})",
    xaxis_title="Pairwise Distance (Cogent3)",
    yaxis_title="MADB Time per kbp (s)",
    width=700,
    height=500,
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=40, r=20, t=40, b=40)
)

fig.show()
fig.write_image(f"../figures/madb_sec_per_kbp_vs_divergence_k{kmer}.pdf")


In [13]:
import pandas
import numpy

# Load the data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Filter to relevant rows
df = df[
    df["species"].isin(["Chimpanzee", "Macaque"]) &
    df["kmer_size"].eq(55) &
    df["madb_time"].notnull() &
    df["cogent3_pd"].notnull()
].copy()

# Compute quartiles for cogent3_pd per species
results = []
for species, group in df.groupby("species"):
    q1 = group["cogent3_pd"].quantile(0.25)
    q4 = group["cogent3_pd"].quantile(0.75)

    low = group[group["cogent3_pd"] <= q1].copy()
    high = group[group["cogent3_pd"] >= q4].copy()

    for label, subset in [("Low (≤ Q1)", low), ("High (≥ Q4)", high)]:
        results.append({
            "Species": species,
            "Divergence Group": label,
            "N": len(subset),
            "Mean Time": subset["madb_time"].mean(),
            "Median Time": subset["madb_time"].median(),
            "Std Dev": subset["madb_time"].std()
        })

quartiles_df = pandas.DataFrame(results)
quartiles_df


,Species,Divergence Group,N,Mean Time,Median Time,Std Dev
0,Chimpanzee,Low (≤ Q1),5,0.172617,0.045334,0.202914
1,Chimpanzee,High (≥ Q4),5,0.182492,0.120042,0.187031
2,Macaque,Low (≤ Q1),13,0.245666,0.123243,0.257438
3,Macaque,High (≥ Q4),13,0.737142,0.566920,0.619464


In [14]:
import pandas
from scipy.stats import ttest_ind, mannwhitneyu

# Load the latest data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Filter to Chimpanzee and Macaque only
df = df[df["species"].isin(["Chimpanzee", "Macaque"])].copy()

# Restrict to k=55 and non-null MADB time and cogent3_pd
kmer = 55
df = df[
    (df["kmer_size"] == kmer) &
    df["madb_time"].notnull() &
    df["cogent3_pd"].notnull()
].copy()

# Compute Q1 and Q4 for each species separately
results = []
for species in ["Chimpanzee", "Macaque"]:
    sub_df = df[df["species"] == species].copy()
    q1 = sub_df["cogent3_pd"].quantile(0.25)
    q4 = sub_df["cogent3_pd"].quantile(0.75)

    low = sub_df[sub_df["cogent3_pd"] <= q1]["madb_time"]
    high = sub_df[sub_df["cogent3_pd"] >= q4]["madb_time"]

    # Perform both parametric and non-parametric tests
    t_stat, t_pval = ttest_ind(low, high, equal_var=False)
    u_stat, u_pval = mannwhitneyu(low, high, alternative="two-sided")

    results.append({
        "Species": species,
        "Low N": len(low),
        "High N": len(high),
        "t-test p-value": t_pval,
        "Mann-Whitney U p-value": u_pval
    })

pandas.DataFrame(results)


,Species,Low N,High N,t-test p-value,Mann-Whitney U p-value
0,Chimpanzee,5,5,0.938205,0.841270
1,Macaque,13,13,0.017755,0.024045
